[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chreissel/sbi-tutorial-iaifi26/blob/main/03_hackathon_stellar_streams.ipynb)

# Notebook 3 — Milky-Way stellar streams (hackathon)

**IAIFI Summer School · simulation-based inference · full-day hackathon.**

Notebooks 1 and 2 built the machinery: a flow posterior, a learned data
embedding, and `falcon`'s "the model is a graph in YAML" workflow. This notebook
points all of it at a real astrophysics problem and then hands you the keys.

A **stellar stream** is what is left when a star cluster or dwarf galaxy is
torn apart by the Milky Way's tides: its stars stretch into a thin ribbon
tracing the orbit. The shape, width, and kinematics of that ribbon encode the
progenitor (its mass and age) *and* the Galactic potential the stream fell
through — which is why streams are one of our best handles on the distribution
of **dark matter**. We use [`sstrax`](https://github.com/undark-lab/sstrax), a
`jax` simulator of the **GD1** stream, wrapped for `falcon` exactly the way
notebook 2 wrapped the gravitational-wave chirp.

This notebook gets you to a **working baseline** — infer a progenitor parameter
or two from a binned image of the stream — in a few cells. The open-ended
challenge is the **hackathon prompt at the end of the repo README**; everything
here is the on-ramp.

> ### ▶️ Run this cell first
> **Runtime → Change runtime type → T4 GPU** helps the flow/CNN training, but
> the simulator itself is CPU-bound `jax`, so a CPU runtime works too. Like
> notebook 2 we **clone** the repo (falcon reads configs and writes samples to
> disk) and install the physics package `sstrax`.
>
> **If you forked or renamed this repository**, change `chreissel/sbi-tutorial-iaifi26`
> below and in the Colab badge above to your own `USER/REPO`.
>
> The install pulls `jax` + `diffrax`; `sstrax` is a git package (not on PyPI).
> The pinned `diffrax` silences a deprecation warning it triggers.

In [ ]:
!git clone -q https://github.com/chreissel/sbi-tutorial-iaifi26.git
%cd sbi-tutorial-iaifi26
!pip install -q -r requirements.txt
# sstrax (the GD1 stream simulator) is a git package, not on PyPI; falcon's Ray
# workers import our streams_model.py which imports it, so it must be installed.
!pip install -q "diffrax==0.6.1" "git+https://github.com/undark-lab/sstrax.git"

In [ ]:
import warnings; warnings.filterwarnings("ignore")   # sstrax/diffrax are chatty
import time
import numpy as np
import matplotlib.pyplot as plt

import streams_model as sm        # the simulator, wrapped for falcon
import streams_plotting as sp     # matplotlib-only plotting helpers

SEED = 0
np.random.seed(SEED)

---
## 0 — A stellar stream, in one simulator call

`sstrax.simulate_stream` takes **16 parameters** (`sm.PRIOR_LIST`) — the
progenitor's present-day position and velocity, its disruption age and mass,
and eight tidal-stripping "micro" parameters — and integrates the disrupting
cluster forward, returning the phase-space coordinates of the stream stars in
the Milky-Way (`halo`) frame:

`stars.shape == (N_stars, 6)` = (x, y, z, vx, vy, vz).

`N_stars` is **not fixed** — older or heavier progenitors shed more stars (a few
hundred to ~1000 here). The **first** call spends ~13 s compiling the `jax`
simulator; every call after that is fast.

In [ ]:
params = sm.sstrax.Parameters()                 # the 16 parameters, at their defaults

t0 = time.time()
stars = np.asarray(sm.sstrax.simulate_stream(key=sm.jax.random.PRNGKey(0), params=params))
print(f"first call : {time.time()-t0:5.1f} s  (includes ~13 s JIT compile)")
t0 = time.time()
_ = np.asarray(sm.sstrax.simulate_stream(key=sm.jax.random.PRNGKey(1), params=params))
print(f"second call: {time.time()-t0:5.2f} s")
print("stars.shape:", stars.shape, "-> (N_stars, 6)")

**Two ways to look at one stream.** On the left, the stars in the Galactic
(`halo`) frame — the physical ribbon wrapping around the Galactic centre. On
the right, the same stars in **GD1 stream coordinates** `(phi1, phi2)`: a
rotated sky frame in which the stream lies flat along `phi1`. That rotation
(`sm.stars_to_gd1`) is a fixed coordinate change we do in plain numpy — see the
note in the next section on why that matters for speed.

In [ ]:
sp.plot_stream_orbit_and_sky(stars, title="one GD1 stream at the fiducial parameters")
plt.show()

---
## 1 — From a variable star list to a fixed-size image

A network cannot read a list whose length changes every simulation. So, exactly
as `albatross` does, we turn each stream into a **fixed-shape image**:

1. rotate to GD1 observables — `(dist, phi1, phi2, vrad, pm_phi1_cosphi2, pm_phi2)`;
2. add per-observable Gaussian measurement errors and drop a few stars (`add_noise`);
3. add a uniform foreground of Milky-Way field stars (`sample_background`);
4. bin into **three 2-D histograms** — `(phi1, phi2)`, the proper motions, and
   `(dist, vrad)` — stacked into a single `(3, nbins, nbins)` array (`bin_stream`).

That fixed `(3, 48, 48)` shape is the whole trick: no matter how many stars a
stream has, the data the network sees is the same size.

> **Why this is fast enough to do live.** `albatross` rotates to GD1 with two
> jitted `jax` vmaps. Because `N_stars` changes every call, those vmaps
> *re-compile every single simulation* — several seconds each. `sm.stars_to_gd1`
> reimplements the identical rotation in numpy (it is affine — a rotation plus
> unit conversions), reproducing `sstrax` to float precision at ~0.5 ms. That
> ~6× speedup is what makes a full training run feasible in a hackathon.
> The timing test is in `hackathon_solutions/`.

In [ ]:
rng = np.random.default_rng(SEED)
truth_full = [sm.TRUE_VALUES[k] for k in sm.TRUE_VALUES]         # all 16 at truth
image = sm.simulate_image(truth_full, infer_params=list(sm.TRUE_VALUES), rng=rng)
print("image shape:", image.shape, " counts per channel:", image.sum((1,2)).astype(int))
sp.plot_channels(image, title="the three data channels a stream maps to")
plt.show()

### ✏️ EXERCISE 1 — watch the parameters move the data

Before inferring anything, build intuition for what is learnable. Simulate the
stream image at **two different disruption ages** (say 1000 and 4000 Myr),
holding everything else at truth, and plot both with `sp.plot_channels`. You
should see the older stream is **longer** along `phi1`.

`sm.simulate_image(z, infer_params=names, rng=...)` takes a value vector `z`
paired with parameter `names`; here vary just `"age"`.

In [ ]:
# TODO — your code here.
#   for age in [1000., 4000.]:
#       img = sm.simulate_image([age], infer_params=["age"],
#                               rng=np.random.default_rng(1))
#       sp.plot_channels(img, title=f"age = {age:.0f} Myr"); plt.show()
raise NotImplementedError("simulate and plot the stream image at two ages")

---
## 2 — The forward model as a `falcon` graph

Same shape as notebook 2. The graph has two nodes:

- **`z`** — the parameters we infer, with a `falcon.priors.Product` prior and a
  `falcon.estimators.Flow`. Its `embedding` is `sm.StreamCNN`, a small 2-D CNN
  that compresses the `(3, 48, 48)` image to a feature vector.
- **`x`** — the data node, produced by `sm.StreamImage` running the full
  forward model on `z` (`parents: [z]`), with `observed:` pointing at an image
  we save to disk.

`streams_model.py` is the thin glue falcon imports in its workers (`sm.StreamImage`
is the simulator, `sm.StreamCNN` the embedding). We start with the smallest
interesting inference: the progenitor's **age and mass**, `sm.DEFAULT_INFER ==
['age', 'logmsat']` — both visibly reshape the stream, so the 2-D posterior is easy to
read. First, save the observation.

In [ ]:
z_truth = [sm.TRUE_VALUES[p] for p in sm.DEFAULT_INFER]     # [age, logmsat] at truth
x_obs = sm.simulate_image(z_truth, rng=np.random.default_rng(42))
np.save("obs_stream.npy", x_obs.astype(np.float32))
print("saved obs_stream.npy", x_obs.shape, "| inferring", sm.DEFAULT_INFER,
      "| truth", z_truth)

### ✏️ EXERCISE 2 — write the falcon config

Fill in the two `# TODO`s below: the **priors** for `age` and `logmsat` (use the
ranges in `sm.PRIOR_RANGES`) and the **`observed:`** path (the file you just
saved). Everything else — the `Flow`, the `StreamCNN` embedding, the buffer — is
already wired against falcon's real schema, the same one you filled in notebook
2. `%%writefile` drops it on disk where falcon's workers can read it.

In [ ]:
%%writefile config_streams.yml
logging:
  wandb: {enabled: false}
  local: {enabled: true}

paths:
  imports: ["."]                    # import streams_model.py from the working dir

buffer:
  min_samples: 256                  # sstrax is ~0.5-1 s/sim, so keep the budget modest
  max_samples: 512
  validation_samples: 96
  simulate_count: 256
  simulate_when_full: false

graph:
  z:
    evidence: [x]
    simulator:
      _target_: falcon.priors.Product
      priors:
        # TODO — two uniform priors for [age, logmsat], from sm.PRIOR_RANGES
        #   age in [500, 5000] Myr, logmsat in [3.0, 4.5]
        #   format is ['uniform', low, high]
        - ['uniform', 0.0, 1.0]
        - ['uniform', 0.0, 1.0]
    estimator:
      _target_: falcon.estimators.Flow
      max_epochs: 40
      net_type: nsf
      lr: 0.003
      gamma: 0.5
      embedding:                    # 2-D CNN over the (3, nbins, nbins) image
        _target_: streams_model.StreamCNN
        out_features: 16
        _input_: [x]
      batch_size: 64
      early_stop_patience: 20
      theta_norm: true
      use_best_models: true
    ray:
      num_gpus: 0

  x:
    parents: [z]
    simulator:
      _target_: streams_model.StreamImage    # infer_params defaults to sm.DEFAULT_INFER
    # TODO — point this at the observation you saved above
    observed: "./TODO_CHANGE_ME.npy"

sample:
  posterior:
    n: 800

**The YAML is the graphical model.** Ask falcon to draw it:

In [ ]:
!falcon graph -c config_streams.yml

---
## 3 — Train, and read out a posterior

Two shell commands, like notebook 2: `launch` simulates ↔ trains the flow, then
we draw posterior samples. The cost is the **simulations** — a few hundred at
~0.5–1 s each, so expect **~10 minutes** on a Colab CPU. (Fewer `min_samples`
for a quick look; more for a sharper posterior — that trade-off is the whole
game, and the hackathon prompt asks you to measure it.)

In [ ]:
!falcon launch -c config_streams.yml -o output/run_streams --no-interactive
!falcon sample posterior -c config_streams.yml -o output/run_streams

In [ ]:
def load_falcon_posterior(run_dir):
    """Stack the per-sample NPZs falcon writes (one (D,) array under key 'z')."""
    import glob
    files = sorted(glob.glob(f"{run_dir}/samples/posterior/*.npz"))
    return np.concatenate([np.atleast_2d(np.load(f)["z"]) for f in files], axis=0)


post = load_falcon_posterior("output/run_streams")
truth = [sm.TRUE_VALUES[p] for p in sm.DEFAULT_INFER]
print("posterior:", post.shape)
sp.plot_posterior(post, sm.DEFAULT_INFER, truth=truth,
                  title="falcon posterior vs truth")
plt.show()

With only a few hundred simulations the posterior is broad, but it should
already **pull toward the truth** and show the physical **age–mass degeneracy**
(a longer stream can mean older *or* heavier). Tightening it — more simulations,
a better embedding, a sequential zoom — is exactly what the hackathon is about.

---
## 4 — The hackathon

You now have a working stream → image → `falcon` posterior loop. **The
open-ended challenge is the "Hackathon prompt" at the end of the repo
[README](https://github.com/chreissel/sbi-tutorial-iaifi26#hackathon-prompt).** Some directions, each a
change of *config or embedding*, not a rewrite:

- **Infer more parameters.** Grow the `Product` prior and the data node's
  `infer_params` to the progenitor's full 6-D phase space, or all 16. Which
  parameters does the stream actually constrain, and which stay at prior?
- **Marginalise the nuisances.** The 8 tidal-stripping micro-parameters are
  uncertain physics you do not care about. Put them in a **second node with a
  prior but no `evidence:`** and falcon integrates them out — notebook 2's
  nuisance pattern. `sm.StreamImage` already accepts a `nuisance_params` list.
- **Design the embedding.** `sm.StreamCNN` is deliberately small. Does a bigger
  CNN, per-channel normalisation, or a different `out_features` buy you a
  tighter posterior at the same simulation budget?
- **Check calibration.** Are the posteriors trustworthy? Run the SBC / coverage
  diagnostics from notebook 1 on a handful of held-out streams.

Compare honestly on **simulation budget** — the number of `sstrax` calls — the
same currency notebook 2 used. Good luck.